# IHMM Demonstration Notebook

This notebook demonstrates how to use the IHMM implementation for fixation detection, including:
- Generating synthetic data
- Running the IHMM pipeline
- Visualizing results


In [1]:
import numpy as np
import matplotlib.pyplot as plt

# import your functions
from pymovements.events.detection.ihmm import ihmm, compute_hmm

import pymovements as pm
from pymovements.gaze.experiment import Experiment
import polars as pl

%config InlineBackend.figure_format = 'svg'

## Load Toy Dataset

In [2]:
# Define the experimental setup
experiment = Experiment(
    screen_width_px=1280,
    screen_height_px=1024,
    screen_width_cm=38,
    screen_height_cm=30.2,
    distance_cm=68,
    origin="upper left",
    sampling_rate=250.0,
)

# Load gaze data from a CSV file and initialize three Gaze objects to test different modes.
gaze1 = pm.gaze.from_csv(
    "./gaze-toy-example.csv",
    experiment=experiment,
    time_column="time",
    pixel_columns=["x", "y"],
)

gaze2 = pm.gaze.from_csv(
    "./gaze-toy-example.csv",
    experiment=experiment,
    time_column="time",
    pixel_columns=["x", "y"],
)
gaze3 = pm.gaze.from_csv(
    "./gaze-toy-example.csv",
    experiment=experiment,
    time_column="time",
    pixel_columns=["x", "y"],
)

# Convert pixel coordinates to degrees of visual angle (dva).
# Requires a valid Experiment with screen geometry and distance.
gaze1.pix2deg()
gaze1.pos2vel()

gaze2.pix2deg()
gaze2.pos2vel()

gaze3.pix2deg()
gaze3.pos2vel()

gaze1

time,pixel,position,velocity
i64,list[f64],list[f64],list[f64]
0,"[206.8, 152.4]","[-10.697598, -8.852399]","[null, null]"
4,"[207.0, 151.5]","[-10.692768, -8.874233]","[null, null]"
8,"[207.6, 151.9]","[-10.678275, -8.86453]","[1.610284, -0.101097]"
12,"[207.6, 152.2]","[-10.678275, -8.857252]","[1.107104, -0.909672]"
16,"[207.8, 151.6]","[-10.673444, -8.871807]","[0.6039, -1.617239]"
…,…,…,…
17204,"[349.3, 420.0]","[-7.220662, -2.272553]","[29.879682, -16.852411]"
17208,"[362.7, 418.1]","[-6.89053, -2.319691]","[43.841686, -7.339814]"
17212,"[371.2, 419.0]","[-6.680877, -2.297363]","[9.957777, -7.442396]"


# The function parameters are as such
```
def ihmm(
        positions: list[list[float]] | list[tuple[float, float]] | np.ndarray,
        timesteps: list[int] | np.ndarray | None = None,
        mu: list[float] | np.ndarray | None = None,
        sigma: list[float] | np.ndarray | None = None,
        init_state: list[float] | np.ndarray | None = None,
        transition_probabilities: list[list[float]] | np.ndarray | None = None,
        reestimation_max_iters: int = 100,
        initialization: str | None = None,
        verbose: bool = False,
        name: str = 'fixation',
) -> Events:
```

## Run IHMM with default parameters

In [3]:
gaze1.detect("ihmm",name="fixation_ihmm")

gaze1.events


[nan nan]


IndexError: index -1 is out of bounds for axis 0 with size 0

# Run IHMM with reestimation

In [ ]:
# Use the "initialization" parameter to specify run the Baum-Welch algorithm to obtain optimal starting parameters
# Flag "verbose" as True to print and see the optimal obtained parameters
# The parameter "reestimation_max_iters" is used to stop the Baum-Welch algorithm if it runs too long,
# it should be need only for very large datasets as the algorithm usually reached convergence quickly 

gaze2.detect("ihmm", initialization = "reestimation", verbose = True, reestimation_max_iters = 100, name="fixation_ihmm")

gaze2.events

In [ ]:
mu= [10, 300]

sigma= [11, 386]

init= [0.5, 0.5] 

trans= [[0.95, 0.05], [0.05, 0.95]] 
    

gaze3.detect("ihmm", mu=mu, sigma=sigma, init_state = init, transition_probabilities = trans, name="fixation_ihmm")

gaze3.events

## Visualize Detected events

In [ ]:
# visualize the data
pm.plotting.traceplot(gaze1)

In [ ]:
gaze1.compute_event_properties(("location", {"position_column": "pixel"}))

pm.plotting.scanpathplot(gaze1, event_name="fixation_ihmm")

In [ ]:
gaze2.compute_event_properties(("location", {"position_column": "pixel"}))

pm.plotting.scanpathplot(gaze2, event_name="fixation_ihmm")

In [ ]:
gaze3.compute_event_properties(("location", {"position_column": "pixel"}))

pm.plotting.scanpathplot(gaze3, event_name="fixation_ihmm")

## Visualize Velocity and States

In [ ]:
# simple velocity computation
velocities = np.sqrt(np.sum(np.diff(positions, axis=0)**2, axis=1))
velocities = np.insert(velocities, 0, 0)

states = compute_hmm(
    velocities=velocities,
    verbose=False,
    initialization='reestimation',
    reestimation_max_iters=1000,
    mu=None,
    sigma=None,
    init_state=None,
    transition_probabilities=None
)

plt.figure()
plt.plot(velocities, label='velocity')
plt.plot(states * np.max(velocities), label='state (scaled)')
plt.legend()
plt.title('Velocity and Inferred States')
plt.show()